In [1]:
import pandas as pd 
import numpy as np 

In [2]:
data = '../models/artifacts/feature_importance_report.csv'

In [3]:
df = pd.read_csv(data)

In [4]:
df = df.sort_values(by ='mean_gain' ,ascending=False)

In [5]:
df.shape

(331, 4)

In [6]:
df.tail(50)

,feature,total_gain,split_count,mean_gain
210,V_pattern_495946718234339098_PCA_7,279.117588,23,12.135547
231,D14,217.817080,18,12.100949
263,V_pattern_6517030450377495652_PCA_15,120.358810,10,12.035881
233,V_pattern_495946718234339098_PCA_8,213.587530,18,11.865974
223,V_pattern_495946718234339098_PCA_14,232.203529,20,11.610176
237,V_pattern_9095697929967337070_PCA_4,190.167032,17,11.186296
283,V_pattern_2390559773846796271_PCA_12,66.582950,6,11.097158
297,D7_normalized,42.411960,4,10.602990
238,V_pattern_495946718234339098_PCA_13,188.552409,18,10.475134
279,V_pattern_1419185338251412470_PCA_15,82.043820,8,10.255478


In [7]:
df.head(50)

,feature,total_gain,split_count,mean_gain
1,V_pattern_2390559773846796271_PCA_8,49986.245401,10,4998.624540
3,V_pattern_959003356291144933_PCA_2,35760.649007,23,1554.810826
17,C7,8702.694374,8,1087.836797
15,V_pattern_1481343147964654334_PCA_1,9475.332342,20,473.766617
10,C8,14684.831512,34,431.906809
8,V_pattern_9095697929967337070_PCA_1,15467.493195,49,315.663126
4,C14,27830.677006,94,296.071032
0,C1,51649.892559,199,259.547199
27,C4,5146.929187,23,223.779530
55,id_17,2482.334143,12,206.861179


In [8]:
# product cd isnot being cast as a categorical feature in the baseline model hence its zero mean gain 
#TransactionAmt_log remove already have TransactionAmt

In [9]:


# Load the generated diagnostic report
# Search for any feature containing "Amt" or "log" to find the target columns
amt_features = df[
    df["feature"].str.contains("Amt|log", case=False, na=False)
]

print("--- Transaction Amount Feature Comparison ---")
display(amt_features)

--- Transaction Amount Feature Comparison ---


,feature,total_gain,split_count,mean_gain
21,addr1_amt_dev_std,7187.455178,169,42.529321
16,uid_amt_mean,9394.783183,285,32.964152
35,TransactionAmt,4296.744016,134,32.065254
33,addr1_amt_dev_median,4631.647857,148,31.294918
23,uid_amt_std,6603.746051,250,26.414984
57,TransactionAmt_cents,2412.918405,96,25.134567


In [11]:
# Search for any feature containing "Amt" or "log" to find the target columns
amt_features = df[
    df["feature"].str.contains("R_email|P_email", case=False, na=False)
]

print("--- Transaction Amount Feature Comparison ---")
display(amt_features)

--- Transaction Amount Feature Comparison ---


,feature,total_gain,split_count,mean_gain
2,R_email_ProductCD,38226.976544,233,164.064277
5,P_email_ProductCD,24399.027768,543,44.933753
49,P_emaildomain,2626.243779,71,36.989349
137,R_emaildomain,794.872074,41,19.387124


In [ ]:
#trial removing the iknteractions 

In [16]:
import lightgbm as lgb
import yaml


df = pd.read_parquet('../data/processed/train_engineered.parquet')

# 2. Isolate ONLY ProductCD to see if it can score splits on its own
X_test = df[["ProductCD"]].copy()
y_test = df["isFraud"]

# 3. Apply the current type conversion
X_test["ProductCD"] = X_test["ProductCD"].astype("category")

# 4. Train a microscopic 5-tree model on ONLY this feature
ds = lgb.Dataset(X_test, label=y_test, categorical_feature=["ProductCD"])
model = lgb.train({"objective": "binary", "verbose": -1}, ds, num_boost_round=5)

# 5. Check the result
standalone_gain = model.feature_importance(importance_type="gain")[0]
print(f"Standalone ProductCD Total Gain: {standalone_gain}")

Standalone ProductCD Total Gain: 46432.68676376343
